# DistilBERT + Retrieval-Augmented Inference

## 1. Setup & Imports

In [ ]:
!pip install -q faiss-gpu sentence-transformers
import os, re, random, warnings
os.environ["WANDB_MODE"] = "disabled"
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit, GroupKFold
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from wordcloud import WordCloud
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import wandb
from sentence_transformers import SentenceTransformer
import faiss
from kaggle_secrets import UserSecretsClient

WANDB_API_KEY = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=WANDB_API_KEY)

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', 100)
sns.set_style("whitegrid")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

LABELS = ['A','B','C','D','E']
DATA_DIR = '/kaggle/input/competitions/smart-mcq-solver-challenge'

## 2. Data Loading & EDA

In [ ]:
train_df = pd.read_csv(os.path.join(DATA_DIR, 'train.csv'))
test_df  = pd.read_csv(os.path.join(DATA_DIR, 'test.csv'))

print(f"Train shape: {train_df.shape}, Test shape: {test_df.shape}")

In [ ]:
plt.figure(figsize=(6,4))
train_df['answer'].value_counts().sort_index().plot(kind='bar', color='skyblue')
plt.title('Answer Distribution (Train)')
plt.xlabel('Option')
plt.ylabel('Count')
plt.show()

In [ ]:
def clean_text(t):
    if pd.isna(t): return ""
    t = str(t).lower()
    t = re.sub(r'[^a-z0-9\s]', ' ', t)
    return re.sub(r'\s+', ' ', t).strip()

def clean_prompt(p):
    p = re.sub(r'(?i)pick the best possible answer:\s*', '', str(p))
    p = re.sub(r'(?i)\s*(among the listed options|from the following choices|carefully)\.?\s*$', '', p)
    return clean_text(p)

def option_set_key(row):
    return tuple(sorted(clean_text(str(row[l])) for l in LABELS))

train_df['prompt_clean'] = train_df['prompt'].apply(clean_prompt)
train_df['option_set'] = train_df.apply(option_set_key, axis=1)
dup_count = train_df.duplicated(subset=['option_set']).sum()
print(f"Number of duplicate option-sets in train: {dup_count}")

In [ ]:
train_df['correct_len'] = train_df.apply(lambda row: len(str(row[row['answer']])), axis=1)
train_df['incorrect_lens'] = train_df.apply(
    lambda row: [len(str(row[l])) for l in LABELS if l != row['answer']], axis=1
)
incorrect_lens = train_df['incorrect_lens'].explode().reset_index(drop=True)
correct_lens = train_df['correct_len']

plot_data = pd.DataFrame({
    'length': pd.concat([correct_lens, incorrect_lens], ignore_index=True),
    'type': ['correct'] * len(correct_lens) + ['incorrect'] * len(incorrect_lens)
})

In [ ]:
plt.figure(figsize=(7, 5))

sns.barplot(
    x='type',
    y='length',
    data=plot_data,
    estimator='mean',
    errorbar=None,
    palette=['green', 'red']
)

plt.title('Average Option Length by Correctness')
plt.xlabel('Correctness')
plt.ylabel('Average Character Length')
plt.show()

In [ ]:
prompt_text = ' '.join(train_df['prompt_clean'].astype(str))
wordcloud = WordCloud(width=800, height=400, background_color='white',
                      max_words=100, random_state=SEED).generate(prompt_text)
plt.figure(figsize=(10,5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.title('Word Cloud of Prompts')
plt.show()

In [ ]:
all_options = []
for _, row in train_df.iterrows():
    for l in LABELS:
        all_options.append(clean_text(row[l]))
tfidf = TfidfVectorizer(max_features=2000, ngram_range=(1,2), sublinear_tf=True)
tfidf.fit(all_options)

sim_matrix = np.zeros((5,5))
for _, row in train_df.iterrows():
    opt_vecs = tfidf.transform([clean_text(row[l]) for l in LABELS])
    sim = cosine_similarity(opt_vecs)
    sim_matrix += sim
sim_matrix /= len(train_df)

plt.figure(figsize=(6,5))
sns.heatmap(sim_matrix, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=LABELS, yticklabels=LABELS)
plt.title('Average Cosine Similarity Between Options')
plt.show()

In [ ]:
prompt_cos_sims = {'correct': [], 'incorrect': []}
for _, row in train_df.iterrows():
    pv = tfidf.transform([clean_prompt(row['prompt'])])
    for l in LABELS:
        ov = tfidf.transform([clean_text(row[l])])
        sim = cosine_similarity(pv, ov)[0,0]
        if l == row['answer']:
            prompt_cos_sims['correct'].append(sim)
        else:
            prompt_cos_sims['incorrect'].append(sim)

plot_data2 = pd.DataFrame({
    'similarity': prompt_cos_sims['correct'] + prompt_cos_sims['incorrect'],
    'type': ['correct'] * len(prompt_cos_sims['correct']) +
            ['incorrect'] * len(prompt_cos_sims['incorrect'])
})

plt.figure(figsize=(7, 5))

sns.barplot(
    x='type',
    y='similarity',
    data=plot_data2,
    estimator='mean',
    errorbar=None,
    palette=['green', 'red']
)

plt.title('Average Cosine Similarity between Prompt and Option')
plt.xlabel('Correctness')
plt.ylabel('Average Cosine Similarity')
plt.ylim(0, 1)
plt.show()

## 2.1 Train–Test Overlap Analysis


In [ ]:
test_df['option_set'] = test_df.apply(option_set_key, axis=1)
train_option_sets = set(train_df['option_set'].unique())
test_matched = test_df['option_set'].isin(train_option_sets)

print(f"Unique option-sets in train: {len(train_option_sets)}")
print(f"Test questions matched to train: {test_matched.sum()} / {len(test_df)} ({test_matched.mean()*100:.1f}%)")
print(f"Test questions unmatched: {(~test_matched).sum()}")

plt.figure(figsize=(5,4))
pd.Series({'Matched': test_matched.sum(), 'Unmatched': (~test_matched).sum()}).plot(
    kind='bar', color=['#2ecc71', '#e74c3c'])
plt.title('Train-Test Overlap (by Option Set)')
plt.ylabel('Count')
plt.show()

## 3. Data Preprocessing & Strict Splitting


In [ ]:
class UnionFind:
    def __init__(self, n): self.p = list(range(n))
    def find(self, x):
        while self.p[x] != x:
            self.p[x] = self.p[self.p[x]]
            x = self.p[x]
        return x
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[ra] = rb

uf = UnionFind(len(train_df))
for col in ['prompt_clean', 'option_set']:
    d = {}
    for i, k in enumerate(train_df[col]):
        if k in d:
            uf.union(i, d[k])
        else:
            d[k] = i
train_df['group_id'] = [uf.find(i) for i in range(len(train_df))]

gss = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
tr_idx, val_idx = next(gss.split(train_df, groups=train_df['group_id']))
train_split = train_df.iloc[tr_idx].reset_index(drop=True)
val_split   = train_df.iloc[val_idx].reset_index(drop=True)

label_map = {l:i for i,l in enumerate(LABELS)}
train_split['label'] = train_split['answer'].map(label_map)
val_split['label']   = val_split['answer'].map(label_map)
y_tr = train_split['label'].values
groups = train_split['group_id'].values

print(f"Train: {len(train_split)}, Val: {len(val_split)} (zero overlap in option-sets)")

## 4. Retrieval Index Setup


In [ ]:
sbert = SentenceTransformer('all-MiniLM-L6-v2', device=str(device))

def build_text(row):
    """Combine prompt + options into a single string for embedding."""
    p = clean_prompt(row['prompt'])
    opts = ' '.join(str(row[l]) for l in LABELS)
    return f"{p} {opts}"

train_texts = train_split.apply(build_text, axis=1).tolist()
test_texts  = test_df.apply(build_text, axis=1).tolist()
val_texts   = val_split.apply(build_text, axis=1).tolist()

print("Encoding training set with SBERT...")
train_embeds = sbert.encode(train_texts, show_progress_bar=True,
                            batch_size=64, normalize_embeddings=True)
test_embeds  = sbert.encode(test_texts,  show_progress_bar=True,
                            batch_size=64, normalize_embeddings=True)
val_embeds   = sbert.encode(val_texts,   show_progress_bar=True,
                            batch_size=64, normalize_embeddings=True)



In [ ]:
# FAISS index
d = train_embeds.shape[1]
index = faiss.IndexFlatIP(d)
index.add(train_embeds.astype(np.float32))
print(f"FAISS index built: {index.ntotal} vectors, dim={d}")

def retrieval_probs(embeds, k=15):
    """For each query, retrieve top-k training neighbors and build
       probability distribution from their answer labels."""
    D, I = index.search(embeds.astype(np.float32), k)
    train_labels = train_split['label'].values
    probs = np.zeros((len(embeds), 5))
    for i in range(len(embeds)):
        for j in range(k):
            sim = max(D[i, j], 0.0)  # cosine similarity
            label = train_labels[I[i, j]]
            probs[i, label] += sim
        total = probs[i].sum()
        if total > 0:
            probs[i] /= total
    return probs

retrieval_probs_test = retrieval_probs(test_embeds)
retrieval_probs_val  = retrieval_probs(val_embeds)
print(f"Retrieval probs shape: {retrieval_probs_test.shape}")

## 5. Metric: MAP@3

In [ ]:
def map3_from_probs(probs, true_idx):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    true_lab = [LABELS[i] for i in true_idx]
    pred_lab = [" ".join(LABELS[j] for j in row) for row in top3]
    score = 0.0
    for t, p in zip(true_lab, pred_lab):
        for i, c in enumerate(p.split()[:3]):
            if c == t:
                score += 1.0/(i+1); break
    return score / len(true_idx)

## 6. DistilBERT Fine-Tuning (5-Fold)

fine-tuning distilbert-base-uncased using AutoModelForMultipleChoice with 5-fold GroupKFold cross-validation

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

sample_len = []
for _, row in train_split.sample(min(300, len(train_split)), random_state=SEED).iterrows():
    p = clean_prompt(row['prompt'])
    for l in LABELS:
        sample_len.append(len(tokenizer(p, row[l])['input_ids']))
MAX_LEN = min(384, int(np.percentile(sample_len, 95)) // 32 * 32 + 32)
print(f"DistilBERT max_length: {MAX_LEN}")

In [ ]:
class HFDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        p = clean_prompt(row['prompt'])
        ids, mask = [], []
        for l in LABELS:
            enc = tokenizer(p, row[l], max_length=MAX_LEN, padding='max_length',
                            truncation=True, return_tensors='pt')
            ids.append(enc['input_ids'].squeeze(0))
            mask.append(enc['attention_mask'].squeeze(0))
        item = {'input_ids': torch.stack(ids), 'attention_mask': torch.stack(mask)}
        if 'label' in row:
            item['labels'] = torch.tensor(row['label'])
        return item

In [ ]:
EPOCHS = 6
LR = 1.5e-5
BATCH_SIZE = 4
GROUP_NAME = "distilbert"

gkf = GroupKFold(n_splits=5)
val_probs_list, test_probs_list = [], []
oof_probs = np.zeros((len(train_split), 5))

for fold, (tr_i, va_i) in enumerate(gkf.split(train_split, y_tr, groups)):
    wandb.init(project="smart-mcq-solver", entity="23f2004192-dl-genai-project",
               group=GROUP_NAME, job_type="cv_fold", name=f"fold_{fold}",
               config={"epochs": EPOCHS, "lr": LR, "max_len": MAX_LEN, "batch_size": BATCH_SIZE})

    tr_df = train_split.iloc[tr_i]
    va_df = train_split.iloc[va_i]
    tr_loader = DataLoader(HFDataset(tr_df), batch_size=BATCH_SIZE, shuffle=True)
    va_loader = DataLoader(HFDataset(va_df), batch_size=BATCH_SIZE, shuffle=False)

    model = AutoModelForMultipleChoice.from_pretrained(MODEL_NAME).to(device)
    opt = optim.AdamW(model.parameters(), lr=LR)
    total_steps = len(tr_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(opt, num_warmup_steps=int(0.1*total_steps),
                                                num_training_steps=total_steps)

    best_map3, best_state = 0, None
    for epoch in range(EPOCHS):
        model.train()
        epoch_loss = 0.0
        for batch in tqdm(tr_loader, desc=f"DB-F{fold} E{epoch}"):
            ids = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)
            loss = model(input_ids=ids, attention_mask=mask, labels=labels).loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step(); scheduler.step(); opt.zero_grad()
            epoch_loss += loss.item()

        model.eval()
        all_probs, all_true = [], []
        with torch.no_grad():
            for batch in va_loader:
                ids = batch['input_ids'].to(device)
                mask = batch['attention_mask'].to(device)
                logits = model(input_ids=ids, attention_mask=mask).logits
                all_probs.append(torch.softmax(logits, dim=-1).cpu().numpy())
                all_true.extend(batch['labels'].cpu().numpy())
            probs_va = np.vstack(all_probs)
            val_map3 = map3_from_probs(probs_va, np.array(all_true))
            val_preds = probs_va.argmax(axis=1)
            val_acc = accuracy_score(np.array(all_true), val_preds)
            val_f1 = f1_score(np.array(all_true), val_preds, average='macro')
            val_loss = nn.CrossEntropyLoss()(torch.tensor(probs_va),
                                             torch.tensor(all_true)).item()

        wandb.log({"epoch": epoch, "train_loss": epoch_loss/len(tr_loader),
                   "val_loss": val_loss, "val_map3": val_map3,
                   "val_accuracy": val_acc, "val_f1": val_f1})

        if val_map3 > best_map3:
            best_map3 = val_map3
            best_state = {k:v.clone() for k,v in model.state_dict().items()}
        print(f"  Fold {fold} E{epoch}: loss={epoch_loss/len(tr_loader):.4f} acc={val_acc:.4f} f1={val_f1:.4f} MAP3={val_map3:.4f}")

    model.load_state_dict(best_state)
    wandb.finish()

    model.eval()
    with torch.no_grad():
        probs = []
        for batch in DataLoader(HFDataset(va_df), batch_size=BATCH_SIZE, shuffle=False):
            ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
            probs.append(torch.softmax(model(input_ids=ids, attention_mask=mask).logits, dim=-1).cpu().numpy())
        oof_probs[va_i] = np.vstack(probs)

        probs = []
        for batch in DataLoader(HFDataset(val_split), batch_size=BATCH_SIZE, shuffle=False):
            ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
            probs.append(torch.softmax(model(input_ids=ids, attention_mask=mask).logits, dim=-1).cpu().numpy())
        val_probs_list.append(np.vstack(probs))

        probs = []
        for batch in DataLoader(HFDataset(test_df), batch_size=BATCH_SIZE, shuffle=False):
            ids = batch['input_ids'].to(device); mask = batch['attention_mask'].to(device)
            probs.append(torch.softmax(model(input_ids=ids, attention_mask=mask).logits, dim=-1).cpu().numpy())
        test_probs_list.append(np.vstack(probs))

In [ ]:
val_probs_bert = np.mean(val_probs_list, axis=0)
test_probs_bert = np.mean(test_probs_list, axis=0)
val_true = val_split['label'].values

bert_val_map3 = map3_from_probs(val_probs_bert, val_true)
print(f"\nDistilBERT-only Val MAP3: {bert_val_map3:.4f}")

## 7. Retrieval-Augmented Inference


In [ ]:
best_alpha, best_map3 = 0, 0
for alpha in np.arange(0.5, 1.0, 0.05):
    blended = alpha * val_probs_bert + (1 - alpha) * retrieval_probs_val
    m3 = map3_from_probs(blended, val_true)
    if m3 > best_map3:
        best_map3 = m3
        best_alpha = alpha
    print(f"  alpha={alpha:.2f}  val_MAP@3={m3:.4f}")

print(f"\nBest alpha: {best_alpha:.2f} -> Val MAP@3: {best_map3:.4f}")

#final predictions
final_test_probs = best_alpha * test_probs_bert + (1 - best_alpha) * retrieval_probs_test
final_val_probs  = best_alpha * val_probs_bert  + (1 - best_alpha) * retrieval_probs_val

def make_submission(probs):
    preds = []
    for i in range(len(probs)):
        top3 = np.argsort(-probs[i])[:3]
        preds.append(' '.join(LABELS[j] for j in top3))
    return preds

test_preds = make_submission(final_test_probs)
print(f"Generated {len(test_preds)} predictions")
